In [ ]:
# Estructura esperada (relativa a la raíz del repo):
#   ./01_data/<proyecto>/AOI/                -> polígonos AOI (incluidos)
#   ./01_data/<proyecto>/images/             -> imágenes Planet (NO incluidas; bajo licencia Planet)
#   ./03_figures_&_results/<proyecto>/        -> tablas derivadas y figuras (salidas)

# Retrospective Phenological Integrity Filter (RPIF) — §3.2
## Spatial Purification of PlanetScope Fixed-Pixel Time Series

**Purpose:**  Remove observations dominated by bare substrate or atmospheric
noise from the NDVI time series, without altering phenologically coherent signal.

**Method:** Builds a one-sided lower-bound envelope from *Gold Standard* years
(structurally stable vegetation, pure pixels ≥ 70 % Veg_Fraction) and rejects
any observation below the weekly threshold. No upper-bound is applied, so
positive anomalies (green flushes) are retained.

**Two variants implemented:**

| Variant | Mode | Gold Standard | Use case |
|---------|------|---------------|----------|
| Retrospective | `backward` | Last *N* stable years | Recovery / succession series (Corema album) |
| Prospective   | `forward`  | First *N* stable years after UAV | Established crops (Avocado, Mango) |

**Data source:** Pre-computed `Pixel_Spectral_Master_v2.parquet` — no raw
image processing. Pixel coverage QC (≥ 99 %) already applied by the pipeline.

**Key outputs:** `Table_RPIF_RejectionStats.csv`, `Table_RPIF_Thresholds_*.csv`,
`Fig7_DataIntegrity.png`, `Fig8_NDVI_Filter_Scatter.png`

---


## 0  Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import ttest_ind, levene, mannwhitneyu
import warnings
warnings.filterwarnings("ignore")

# ─── Paths ────────────────────────────────────────────────────────────────────
BASE = r".\03_figures_&_results"

PROJECTS = {
    "corema_album": {
        "pixel_file":  os.path.join(BASE, "01_corema_album",
                                    "Pixel_Spectral_Master_v2.parquet"),
        "output_dir":  os.path.join(BASE, "01_corema_album"),
        # RETROSPECTIVE: UAV Jun 2025 -> gold standard = last 3 stable H_Years.
        # Pre-2023 data unreliable (post-fire recovery, canopy still consolidating).
        "temporal_mode": "backward",
        "years_gold":   [2023, 2024, 2025],
        "series_start": "2021-10-01",
        "series_end":   "2025-09-30",
        "entities": {"corema_A": "Corema album", "corema_B": "Corema album"},
        "color": "#2874A6",
    },
    "avocado_mango": {
        "pixel_file":  os.path.join(BASE, "02_avocado_mango",
                                    "Pixel_Spectral_Master_v2.parquet"),
        "output_dir":  os.path.join(BASE, "02_avocado_mango"),
        # PROSPECTIVE: UAV Aug 2019 -> use years CLOSEST to drone acquisition
        # (H_Years 2021-2022-2023 are nearest to the purity reference).
        # Established crops: no disturbance, all years valid, but early ones
        # best reflect the canopy state captured by the drone.
        "temporal_mode": "forward",
        "years_gold":   [2021, 2022, 2023, 2024, 2025],
        "series_start": "2020-10-01",
        "series_end":   "2025-09-30",
        "entities": {"avocado": "Avocado", "mango": "Mango"},
        "color": "#27AE60",
    },
}

# ─── RPIF parameters ──────────────────────────────────────────────────────────
VEG_PURITY_MIN      = 0.70   # min Veg_Fraction for Gold Standard training
IQR_TOLERANCE_LOWER = 1.0    # lower_limit = Q25 - 1 * IQR  (one-sided)
SMOOTHING_WINDOW    = 3      # weeks rolling average on envelope

# ─── Output ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(BASE, "00_rpif_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial"],
    "font.size": 12,  "axes.labelsize": 13,
    "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11,
    "axes.grid": False, "figure.dpi": 150,
})

print("Configuration loaded.")
for name, cfg in PROJECTS.items():
    ok = os.path.exists(cfg["pixel_file"])
    print(f"  {name}: parquet={'OK' if ok else 'MISSING'}  mode={cfg['temporal_mode']}")


---
## 1  Load & Preprocess

Loads pre-computed pixel spectral databases (parquet), adds hydrological
week / year, and assigns entity labels. Applies date-range filter per project.

> **QC already applied** by the pipeline: only acquisitions with pixel
> coverage ≥ 99 % are present in the parquet. No jitter filter needed.


In [ ]:
def add_hydro_columns(df):
    # Reset to 0-based index before any assignment to guarantee positional
    # alignment regardless of the DataFrame's original (possibly non-contiguous)
    # index.  Also uses tz_convert (not tz_localize) for tz-aware Series.
    df = df.copy().reset_index(drop=True)

    # Strip timezone -> naive UTC datetime64[ns]
    if df["Date"].dt.tz is not None:
        dates = df["Date"].dt.tz_convert(None)
    else:
        dates = df["Date"].copy()

    # Extract components as plain numpy arrays
    month = dates.dt.month.values
    year  = dates.dt.year.values

    # Hydrological year: months Oct-Dec belong to year+1
    h_year = np.where(month >= 10, year + 1, year).astype(int)

    # Oct 1 of each hydrological year as datetime64[D]
    start_years = np.where(month >= 10, year, year - 1).astype(int)
    oct1_dates  = np.array(
        [f"{int(y)}-10-01" for y in start_years],
        dtype="datetime64[D]"
    )

    # Day-truncated acquisition dates
    dates_day = dates.values.astype("datetime64[D]")

    days   = (dates_day - oct1_dates).astype(int)
    h_week = np.clip((days // 7) + 1, 1, 52).astype(int)

    df["H_Year"] = h_year
    df["H_Week"] = h_week
    return df


# ── Load both projects ────────────────────────────────────────────────────────
all_frames = []

for proj_name, cfg in PROJECTS.items():

    df = pd.read_parquet(cfg["pixel_file"])
    df["Date"] = pd.to_datetime(df["Date"], utc=True)

    # Entity label from AOI
    df["Entity"]  = df["AOI"].map(cfg["entities"]).fillna(df["AOI"])
    df["Project"] = proj_name

    # Date range — use UTC-aware Timestamps to avoid tz-naive/aware mismatch
    t_start = pd.Timestamp(cfg["series_start"], tz="UTC")
    t_end   = pd.Timestamp(cfg["series_end"],   tz="UTC")
    df = df[(df["Date"] >= t_start) & (df["Date"] <= t_end)].copy()

    # Hydrological columns (numpy-based, index-safe after reset_index inside)
    df = add_hydro_columns(df)

    all_frames.append(df)
    h_years  = sorted(df["H_Year"].unique().tolist())
    entities = df["Entity"].unique().tolist()
    print(f"  {proj_name}: {len(df):,} rows  H_Years={h_years}  entities={entities}")

df_all = pd.concat(all_frames, ignore_index=True)

print(f"\nTotal observations loaded: {len(df_all):,}")
print(f"Entities:  {df_all['Entity'].unique().tolist()}")
print(f"H_Year range: {df_all['H_Year'].min()} - {df_all['H_Year'].max()}")


---
## 2  Gold Standard Envelope Construction

For each entity, the phenological envelope is built from:
- **Temporal filter:** `years_gold` defined per project (H_Year).
- **Spatial filter:** pixels with `Veg_Fraction >= 0.70` (geometrically pure).
- **Statistic:** Q25 - 1 * IQR per hydrological week → one-sided lower bound.
- **Smoothing:** 3-week centred rolling mean to reduce week-to-week noise.

**Retrospective (backward, Corema):** last 3 stable H_Years (2023-2025) —
vegetation is structurally mature and pixel purity is highest.

**Prospective (forward, Avocado/Mango):** same algorithm but logically
anchored at the UAV acquisition date. For established crops the distinction
has negligible effect; for recovering vegetation it would use the first
post-UAV stable years.


In [ ]:
def build_envelope(df_entity, years_gold, veg_min=VEG_PURITY_MIN,
                   iqr_tol=IQR_TOLERANCE_LOWER, smooth=SMOOTHING_WINDOW):
    # Training set: pure pixels in Gold Standard years
    mask = (df_entity["H_Year"].isin(years_gold) &
            (df_entity["Veg_Fraction"] >= veg_min))
    df_train = df_entity[mask].copy()

    if df_train.empty:
        print("  [WARNING] No training data — check years_gold or VEG_PURITY_MIN")
        return None

    # Weekly statistics
    grp = df_train.groupby("H_Week")["NDVI"]
    env = grp.agg(
        median="median",
        q25=lambda x: x.quantile(0.25),
        q75=lambda x: x.quantile(0.75),
        n_train="count",
    ).reset_index()
    env["iqr"] = env["q75"] - env["q25"]
    env["lower_limit"] = env["q25"] - iqr_tol * env["iqr"]

    # Smoothing
    env["lower_limit"] = (env["lower_limit"]
                          .rolling(smooth, center=True, min_periods=1)
                          .mean())

    print(f"  Training obs: {len(df_train):,} | "
          f"Weeks covered: {len(env)} | "
          f"Threshold range: [{env['lower_limit'].min():.3f}, "
          f"{env['lower_limit'].max():.3f}]")
    return env


# # Build one envelope per entity
# envelopes = {}

# for proj_name, cfg in PROJECTS.items():
#     df_proj = df_all[df_all["Project"] == proj_name]
#     for entity in df_proj["Entity"].unique():
#         df_ent = df_proj[df_proj["Entity"] == entity]
#         print(f"\n{entity} ({cfg['temporal_mode']} mode, "
#               f"gold={cfg['years_gold']}):")
#         env = build_envelope(df_ent, cfg["years_gold"])
#         envelopes[entity] = (env, cfg)

# print("\nEnvelopes built for:", list(envelopes.keys()))

# Definición de umbrales adaptativos basados en arquitectura del dosel
RPIF_PARAMS = {
    "Avocado": {"iqr_tol": 1.0, "smooth": 3},
    "Corema album": {"iqr_tol": 1.0, "smooth": 3},
    "Mango": {"iqr_tol": 2.0, "smooth": 3}  # Tolerancia relajada para dosel abierto/heterogéneo
}

# Build one envelope per entity
envelopes = {}

for proj_name, cfg in PROJECTS.items():
    df_proj = df_all[df_all["Project"] == proj_name]
    for entity in df_proj["Entity"].unique():
        df_ent = df_proj[df_proj["Entity"] == entity]
        
        # Extraer parámetros específicos. Si hay una entidad nueva, usa 1.0 y 3 por defecto
        params = RPIF_PARAMS.get(entity, {"iqr_tol": 1.0, "smooth": 3})
        
        print(f"\n{entity} ({cfg['temporal_mode']} mode, "
              f"gold={cfg['years_gold']} | IQR_tol={params['iqr_tol']}):")
        
        # Inyectar explícitamente los parámetros en la función
        env = build_envelope(
            df_ent, 
            cfg["years_gold"],
            iqr_tol=params["iqr_tol"],
            smooth=params["smooth"]
        )
        envelopes[entity] = (env, cfg)

print("\nEnvelopes adaptativos built for:", list(envelopes.keys()))


---
## 3  Apply RPIF Filter

Merges each entity's observations with its weekly lower-bound envelope
and sets `is_bio_valid = True` when `NDVI >= lower_limit`.

> **Note on cloud removal (connection §3.1 → §3.2):**  Clouds and cloud shadows
> produce NDVI values near 0 or negative — well below the vegetation threshold
> (typically 0.17–0.20 for the study areas).  The RPIF therefore removes cloud-
> contaminated pixels automatically, without requiring a separate cloud mask.


In [ ]:
filtered_frames = []

for proj_name, cfg in PROJECTS.items():
    df_proj = df_all[df_all["Project"] == proj_name].copy()

    for entity in df_proj["Entity"].unique():
        env_data = envelopes.get(entity)
        if env_data is None:
            continue
        env, _ = env_data

        df_ent = df_proj[df_proj["Entity"] == entity].copy()

        # Merge threshold
        df_merged = df_ent.merge(env[["H_Week", "lower_limit"]],
                                 on="H_Week", how="left")
        df_merged["is_bio_valid"] = df_merged["NDVI"] >= df_merged["lower_limit"]

        filtered_frames.append(df_merged)

df_filtered = pd.concat(filtered_frames, ignore_index=True)

# Quick summary
print("=" * 60)
print("RPIF FILTER APPLIED")
print("=" * 60)
total   = len(df_filtered)
valid   = df_filtered["is_bio_valid"].sum()
removed = total - valid
print(f"  Total observations:  {total:>10,}")
print(f"  Valid (bio_valid):   {valid:>10,}  ({valid/total*100:.2f}%)")
print(f"  Removed:             {removed:>10,}  ({removed/total*100:.2f}%)")
print()
for ename, grp in df_filtered.groupby("Entity"):
    n = len(grp); v = grp["is_bio_valid"].sum(); r = n-v
    print(f"  {ename:<16}  n={n:>8,}  valid={v:>7,} ({v/n*100:.1f}%)  "
          f"removed={r:>6,} ({r/n*100:.1f}%)")


---
## 4  Validation Statistics

Reproduces the metrics reported in §3.2 of the manuscript:
- Rejection rate per hydrological year (survivorship-bias pattern)
- NDVI distribution of removed (substrate noise) vs. valid (vegetation)
- σ reduction after filtering
- Hypothesis tests: Welch t, Levene, Mann-Whitney U


In [ ]:
stat_rows = []
test_rows = []

for entity, grp in df_filtered.groupby("Entity"):

    df_valid   = grp[grp["is_bio_valid"]]["NDVI"].dropna()
    df_removed = grp[~grp["is_bio_valid"]]["NDVI"].dropna()
    df_total   = grp["NDVI"].dropna()

    # ── Descriptive ────────────────────────────────────────────────────────
    std_total = df_total.std()
    std_valid = df_valid.std()
    std_red   = (std_total - std_valid) / std_total * 100

    print(f"\n{'='*55}")
    print(f"  {entity}")
    print(f"{'='*55}")
    print(f"  Total:     n={len(df_total):>7,}  mean={df_total.mean():.4f}"
          f"  sigma={std_total:.4f}")
    print(f"  Valid:     n={len(df_valid):>7,}  mean={df_valid.mean():.4f}"
          f"  sigma={std_valid:.4f}")
    print(f"  Removed:   n={len(df_removed):>7,}  mean={df_removed.mean():.4f}"
          f"  sigma={df_removed.std():.4f}")
    print(f"  sigma reduction: {std_red:.2f}%")

    # ── Hypothesis tests ────────────────────────────────────────────────────
    if len(df_valid) > 0 and len(df_removed) > 0:
        t_stat, t_p   = ttest_ind(df_total, df_valid, equal_var=False)
        lev_stat, l_p = levene(df_total, df_valid)
        mw_stat, m_p  = mannwhitneyu(df_total, df_valid, alternative="two-sided")
        print(f"  Welch t:     t={t_stat:.2f}  p={t_p:.2e}")
        print(f"  Levene:      F={lev_stat:.2f}  p={l_p:.2e}")
        print(f"  Mann-Whitney U={mw_stat:.0f}  p={m_p:.2e}")
    else:
        t_p = l_p = m_p = np.nan

    stat_rows.append({
        "Entity":            entity,
        "n_total":           len(df_total),
        "n_valid":           len(df_valid),
        "n_removed":         len(df_removed),
        "Rejection_%":       round((len(df_removed)/len(df_total))*100, 2),
        "NDVI_total_mean":   round(df_total.mean(), 4),
        "NDVI_valid_mean":   round(df_valid.mean(), 4),
        "NDVI_removed_mean": round(df_removed.mean(), 4) if len(df_removed)>0 else np.nan,
        "Sigma_total":       round(std_total, 4),
        "Sigma_valid":       round(std_valid, 4),
        "Sigma_reduction_%": round(std_red, 2),
        "Welch_p":           t_p, "Levene_p": l_p, "MannWhitney_p": m_p,
    })

# ── Rejection rate per H_Year (survivorship bias table) ──────────────────────
print("\n\n=== REJECTION RATE PER HYDROLOGICAL YEAR ===")
year_rows = []
for (entity, h_year), g in df_filtered.groupby(["Entity","H_Year"]):
    n = len(g); r = (~g["is_bio_valid"]).sum()
    year_rows.append({"Entity": entity, "H_Year": h_year,
                      "n_total": n, "n_removed": r,
                      "Rejection_%": round(r/n*100, 2)})
df_year_stats = pd.DataFrame(year_rows)
display(df_year_stats.pivot_table(index="H_Year", columns="Entity",
                                  values="Rejection_%").round(2))

df_stats = pd.DataFrame(stat_rows)
print()
display(df_stats[["Entity","n_total","n_valid","n_removed",
                  "Rejection_%","NDVI_valid_mean","NDVI_removed_mean",
                  "Sigma_reduction_%"]].round(4))

df_stats.to_csv(os.path.join(OUTPUT_DIR, "Table_RPIF_RejectionStats.csv"), index=False)
df_year_stats.to_csv(os.path.join(OUTPUT_DIR, "Table_RPIF_RejectionByYear.csv"), index=False)
print("\n[SAVED] Table_RPIF_RejectionStats.csv + Table_RPIF_RejectionByYear.csv")


---
## 5  Figure 7 — Dataset Integrity After RPIF

**(A)** Absolute observation count per hydrological year, split by
valid vegetation pixels (green) and removed substrate noise (red).
The survivorship bias pattern is visible in the early H_Years.

**(B)** Monthly distribution across all hydrological years: valid vs removed.


In [ ]:
entity_list = df_filtered["Entity"].unique().tolist()
n_entities  = len(entity_list)

# Colour palette
COLORS = {"valid": "#2ca02c", "removed": "#d62728"}

fig, axes = plt.subplots(2, n_entities,
                          figsize=(5 * n_entities, 10),
                          sharey=False)
if n_entities == 1:
    axes = axes.reshape(2, 1)

plt.rcParams.update({"font.family":"sans-serif","font.sans-serif":["Arial"],
                     "font.size":12,"axes.labelsize":13,"axes.grid":False,
                     "xtick.labelsize":11,"ytick.labelsize":11,"figure.dpi":300})

for col_idx, entity in enumerate(entity_list):

    grp = df_filtered[df_filtered["Entity"] == entity]

    # ── Panel A: per H_Year ───────────────────────────────────────────────
    ax = axes[0, col_idx]

    stats_yr = (grp.groupby("H_Year")["is_bio_valid"]
                .value_counts().unstack(fill_value=0)
                .rename(columns={True: "Valid", False: "Removed"}))
    # Ensure both columns exist
    for c in ["Valid","Removed"]:
        if c not in stats_yr.columns: stats_yr[c] = 0

    stats_yr[["Valid","Removed"]].plot(
        kind="bar", stacked=True,
        color=[COLORS["valid"], COLORS["removed"]],
        ax=ax, alpha=0.85, edgecolor="black", linewidth=0.4,
        legend=(col_idx == 0)
    )

    # Annotate rejection %
    for i, (yr, row) in enumerate(stats_yr.iterrows()):
        tot = row["Valid"] + row["Removed"]
        pct = row["Removed"] / tot * 100 if tot > 0 else 0
        ax.text(i, tot + tot*0.01, f"{pct:.1f}%",
                ha="center", va="bottom", fontsize=8, color=COLORS["removed"])

    ax.set_title(entity, fontweight="bold", fontsize=13)
    ax.set_xlabel("Hydrological Year")
    ax.set_ylabel("Observation Count" if col_idx == 0 else "")
    ax.tick_params(axis="x", rotation=0)
    if col_idx == 0:
        ax.legend(["Valid (Vegetation)", "Removed (Substrate/Cloud)"],
                  frameon=False, fontsize=10)
    ax.text(-0.12, 1.05, "A", transform=ax.transAxes,
            fontsize=18, fontweight="bold")

    # ── Panel B: monthly distribution ─────────────────────────────────────
    ax = axes[1, col_idx]

    grp2 = grp.copy()
    grp2["Month"] = grp2["Date"].dt.month
    stats_mo = (grp2.groupby("Month")["is_bio_valid"]
                .value_counts().unstack(fill_value=0)
                .rename(columns={True: "Valid", False: "Removed"}))
    for c in ["Valid","Removed"]:
        if c not in stats_mo.columns: stats_mo[c] = 0

    # Reindex to ensure all 12 months
    stats_mo = stats_mo.reindex(range(1,13), fill_value=0)

    stats_mo[["Valid","Removed"]].plot(
        kind="bar", stacked=True,
        color=[COLORS["valid"], COLORS["removed"]],
        ax=ax, alpha=0.85, edgecolor="black", linewidth=0.4,
        legend=False
    )
    month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                    "Jul","Aug","Sep","Oct","Nov","Dec"]
    ax.set_xticklabels(month_labels, rotation=45, ha="right")
    ax.set_xlabel("Month")
    ax.set_ylabel("Observation Count" if col_idx == 0 else "")
    ax.text(-0.12, 1.05, "B", transform=ax.transAxes,
            fontsize=18, fontweight="bold")

# plt.suptitle("Figure 7 — RPIF Dataset Integrity Assessment",
#              fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()

out7 = os.path.join(OUTPUT_DIR, "Fig7_DataIntegrity.png")
plt.savefig(out7, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {out7}")


---
## 6  Figure 8 — NDVI Time Series with RPIF Threshold

Scatter of all observations (NDVI vs hydrological week) coloured by
filtering outcome. The dark green dashed line is the Gold Standard lower-bound
envelope; the shaded region is the acceptance zone.


In [ ]:
fig, axes = plt.subplots(1, n_entities,
                          figsize=(7 * n_entities, 6),
                          sharey=True)
if n_entities == 1:
    axes = [axes]

for ax, entity in zip(axes, entity_list):

    grp   = df_filtered[df_filtered["Entity"] == entity]
    env   = envelopes[entity][0]

    valid   = grp[grp["is_bio_valid"]]
    invalid = grp[~grp["is_bio_valid"]]

    # Acceptance zone shading
    ax.fill_between(env["H_Week"], env["lower_limit"], 1.0,
                    color="#2ca02c", alpha=0.07,
                    label="Acceptance zone (phenological reference envelope)")

    # Scatter
    ax.scatter(valid["H_Week"],   valid["NDVI"],
               c="#2ca02c", s=5, alpha=0.25, linewidths=0, label="Valid")
    ax.scatter(invalid["H_Week"], invalid["NDVI"],
               c="#d62728", s=12, marker="x", alpha=0.55,
               linewidths=0.8, label="Removed (substrate / cloud)")

    # Envelope
    ax.plot(env["H_Week"], env["lower_limit"],
            color="darkgreen", linestyle="--", linewidth=2,
            label="Lower threshold (RPIF)")

    # Annotate envelope range
    ax.text(0.02, 0.97,
            f"Threshold: {env['lower_limit'].min():.3f}–{env['lower_limit'].max():.3f}",
            transform=ax.transAxes, fontsize=9, va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      alpha=0.85, edgecolor="#bbb"))

    ax.set_xlim(0, 53); ax.set_ylim(-0.05, 1.0)
    ax.set_xlabel("Hydrological Week (1 = 1 Oct)", fontsize=12)
    ax.set_ylabel("NDVI" if entity == entity_list[0] else "")
    ax.set_title(entity, fontweight="bold", fontsize=13)

    if entity == entity_list[0]:
        ax.legend(loc="lower right", fontsize=9, frameon=True,
                  framealpha=0.9)

# plt.suptitle("Figure 8 — Application of RPIF to the NDVI Time Series",
#              fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()

out8 = os.path.join(OUTPUT_DIR, "Fig8_NDVI_Filter_Scatter.png")
plt.savefig(out8, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {out8}")


---
## 7  NDVI Distribution: Valid vs Removed

KDE comparison confirms that the removed population clusters around bare substrate
reflectance values, while the valid population represents active vegetation.


In [ ]:
fig, axes = plt.subplots(1, n_entities, figsize=(6 * n_entities, 5), sharey=False)
if n_entities == 1: axes = [axes]

for ax, entity in zip(axes, entity_list):

    grp     = df_filtered[df_filtered["Entity"] == entity]
    valid   = grp[grp["is_bio_valid"]]["NDVI"].dropna()
    removed = grp[~grp["is_bio_valid"]]["NDVI"].dropna()
    total   = grp["NDVI"].dropna()

    sns.kdeplot(total,   ax=ax, color="gray",     linewidth=1.5,
                linestyle=":", label="Total")
    sns.kdeplot(valid,   ax=ax, color="#2ca02c",  linewidth=2.0,
                label=f"Valid (mean={valid.mean():.3f})")
    if len(removed) > 5:
        sns.kdeplot(removed, ax=ax, color="#d62728", linewidth=2.0,
                    linestyle="--",
                    label=f"Removed (mean={removed.mean():.3f})")

    ax.axvline(valid.mean(),   color="#2ca02c", linestyle="--",
               linewidth=1, alpha=0.7)
    ax.axvline(removed.mean(), color="#d62728", linestyle="--",
               linewidth=1, alpha=0.7)

    ax.set_xlabel("NDVI")
    ax.set_ylabel("Density" if entity == entity_list[0] else "")
    ax.set_title(entity, fontweight="bold")
    ax.legend(fontsize=9, frameon=True)

# plt.suptitle("NDVI Distribution: Valid vs Removed Populations",
#              fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()

out_kde = os.path.join(OUTPUT_DIR, "Fig_NDVI_Distribution_ValidRemoved.png")
plt.savefig(out_kde, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {out_kde}")


In [ ]:
fig, axes = plt.subplots(1, n_entities, figsize=(6 * n_entities, 5), sharey=False)
if n_entities == 1: axes = [axes]

for ax, entity in zip(axes, entity_list):

    grp     = df_filtered[df_filtered["Entity"] == entity]
    valid   = grp[grp["is_bio_valid"]]["NDVI"].dropna()
    removed = grp[~grp["is_bio_valid"]]["NDVI"].dropna()
    total   = grp["NDVI"].dropna()

    sns.kdeplot(total,   ax=ax, color="gray",     linewidth=1.5,
                linestyle=":", label="Total")
    sns.kdeplot(valid,   ax=ax, color="#2ca02c",  linewidth=2.0,
                label=f"Valid (mean={valid.mean():.3f})")
    if len(removed) > 5:
        sns.kdeplot(removed, ax=ax, color="#d62728", linewidth=2.0,
                    linestyle="--", label=f"Removed (mean={removed.mean():.3f})")

    ax.axvline(valid.mean(),   color="#2ca02c", linestyle="--", linewidth=1, alpha=0.7)
    ax.axvline(removed.mean(), color="#d62728", linestyle="--", linewidth=1, alpha=0.7)

    # --- Mensaje incrustado de la antigua Fig 14: fracción rechazada + peso en varianza ---
    n_tot   = len(total)
    frac    = len(removed) / n_tot if n_tot else np.nan
    var_tot = float(total.var())
    var_val = float(valid.var())
    var_red = (1.0 - var_val / var_tot) if var_tot > 0 else np.nan   # varianza eliminada por el filtro
    ax.text(0.97, 0.97,
            f"Removed: {frac:.1%} of obs\n\u2192 {var_red:.0%} variance reduction",
            transform=ax.transAxes, fontsize=8.5, va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      alpha=0.85, edgecolor="#bbb"))

    ax.set_xlabel("NDVI")
    ax.set_ylabel("Density" if entity == entity_list[0] else "")
    ax.set_title(entity, fontweight="bold")
    ax.legend(fontsize=9, frameon=True, loc="upper left")

plt.tight_layout()
out_merged = os.path.join(OUTPUT_DIR, "Fig_NDVI_Distribution_merged.png")
plt.savefig(out_merged, dpi=300, bbox_inches="tight")
plt.show()
print(f"[SAVED] {out_merged}")

---
## 8  Export

Threshold tables (one per entity) for the manuscript appendix, and the
filtered master dataset for downstream analysis.


In [ ]:
# ── Threshold tables ──────────────────────────────────────────────────────────
print("=== RPIF THRESHOLD TABLES ===")

for entity, (env, cfg) in envelopes.items():
    tbl = env[["H_Week","lower_limit","median","q25","q75","n_train"]].copy()
    tbl.columns = ["H_Week","Lower_Limit","Median","Q25","Q75","n_train_obs"]
    tbl = tbl.round(4)

    fname = f"Table_RPIF_Thresholds_{entity.replace(' ','_')}.csv"
    out   = os.path.join(OUTPUT_DIR, fname)
    tbl.to_csv(out, index=False)
    print(f"  [SAVED] {fname}")
    display(tbl.head(8))
    print()


# ── Filtered master dataset ────────────────────────────────────────────────────
cols_export = ["Entity","Project","AOI","pixel_id","Date","H_Year","H_Week",
               "Veg_Fraction","NDVI","NDRE","GNDVI","PRI_proxy",
               "Red","NIR","Green_I","Yellow", "Green", "RedEdge","Coastal_Blue", "Blue",
               "True_Solar_Hour","Solar_Zenith","View_Angle",
               "lower_limit","is_bio_valid"]
cols_export = [c for c in cols_export if c in df_filtered.columns]

df_out = df_filtered[cols_export].copy()

# Parquet (fast, lossless)
parquet_out = os.path.join(OUTPUT_DIR, "Master_RPIF_Filtered.parquet")
df_out.to_parquet(parquet_out, index=False, compression="zstd")
print(f"[SAVED] Master_RPIF_Filtered.parquet  ({len(df_out):,} rows)")

# Summary
print()
print("=" * 60)
print("FINAL FILTERED DATASET SUMMARY")
print("=" * 60)
for entity, g in df_out.groupby("Entity"):
    valid   = g["is_bio_valid"].sum()
    n       = len(g)
    n_dates = g["Date"].nunique()
    n_pix   = g["pixel_id"].nunique()
    print(f"  {entity:<16}  n={n:>8,}  valid={valid:>7,} ({valid/n*100:.1f}%)  "
          f"dates={n_dates:>4}  pixels={n_pix:>4}")

print()
print(f"All outputs in: {OUTPUT_DIR}")


---
## Summary of outputs

```
00_rpif_output/
  Table_RPIF_RejectionStats.csv          <- overall rejection stats + hypothesis tests
  Table_RPIF_RejectionByYear.csv         <- rejection rate per H_Year (survivorship pattern)
  Table_RPIF_Thresholds_<entity>.csv     <- weekly NDVI lower-bound (paper appendix)
  Master_RPIF_Filtered.parquet           <- filtered pixel database for downstream use
  Fig7_DataIntegrity.png                 <- paper Figure 7
  Fig8_NDVI_Filter_Scatter.png           <- paper Figure 8
  Fig_NDVI_Distribution_ValidRemoved.png <- KDE comparison
```

**Connection §3.1 → §3.2:** Cloud-contaminated pixels (NDVI ~ 0) fall below the
weekly vegetation threshold (typically 0.17–0.20) and are removed by the RPIF
automatically, without a separate cloud filter. This links the radiometric
instability identified in §3.1 to the purification workflow in §3.2.


In [ ]:
import os, numpy as np, pandas as pd
BANDS = ["Coastal_Blue","Blue","Green_I","Green","Yellow","Red","RedEdge","NIR"]
BANDS = [b for b in BANDS if b in df_filtered.columns]

rows_sig, rows_cls = [], []
for entity, g in df_filtered.groupby("Entity"):
    rem, val = g[~g["is_bio_valid"]], g[g["is_bio_valid"]]
    if len(rem) == 0: continue
    sig = {"Entity": entity, "n_removed": len(rem)}
    for b in BANDS:
        sig[f"{b}_rem"] = round(rem[b].mean(), 3)
        sig[f"{b}_val"] = round(val[b].mean(), 3)
    rows_sig.append(sig)
    # Clasificación heurística del PÍXEL eliminado (umbrales para reflectancia 0-1; ajústalos a tu escala):
    nir, red, blue = rem["NIR"], rem["Red"], rem["Blue"]
    cloud = (blue > 0.18) | (nir > 0.34)        # nube/bruma: brillante (azul o NIR altos)
    soil  = (~cloud) & (red > 0.13)             # suelo/arena: rojo moderado, no brillante
    other = ~(cloud | soil)
    rows_cls.append({"Entity": entity, "n_removed": len(rem),
                     "cloud_like_%": round(cloud.mean()*100, 1),
                     "soil_like_%":  round(soil.mean()*100, 1),
                     "other_%":      round(other.mean()*100, 1)})

sig_df, cls_df = pd.DataFrame(rows_sig), pd.DataFrame(rows_cls)
print("=== Clasificación de lo eliminado ==="); print(cls_df.to_string(index=False))
print("\n=== Firma espectral media (eliminado vs válido) ==="); display(sig_df)
sig_df.to_csv(os.path.join(OUTPUT_DIR, "Table_RPIF_RemovedSpectralSignature.csv"), index=False)
cls_df.to_csv(os.path.join(OUTPUT_DIR, "Table_RPIF_RemovedClassification.csv"), index=False)

In [ ]:
import os, numpy as np, pandas as pd
from scipy.stats import mannwhitneyu

meta_frames = []
for proj_name, cfg in PROJECTS.items():
    mp = os.path.join(cfg["output_dir"], "Metadata_Master_NOAA.csv")
    if os.path.exists(mp):
        m = pd.read_csv(mp)
        keep = [c for c in ["Smart_ID","Cloud_Percent","Unusable_Percent"] if c in m.columns]
        if "Smart_ID" in keep: meta_frames.append(m[keep].drop_duplicates("Smart_ID"))

if not meta_frames:
    print("[SKIP] No hay Metadata_Master_NOAA.csv -> no se puede cruzar con nube.")
else:
    meta = pd.concat(meta_frames, ignore_index=True).drop_duplicates("Smart_ID")
    dfm  = df_filtered.merge(meta, on="Smart_ID", how="left")
    rows = []
    for entity, g in dfm.groupby("Entity"):
        for col in ["Cloud_Percent","Unusable_Percent"]:
            if col not in g.columns: continue
            rem = g.loc[~g["is_bio_valid"], col].dropna()
            val = g.loc[g["is_bio_valid"],  col].dropna()
            if len(rem) < 20 or len(val) < 20: continue
            u, p = mannwhitneyu(rem, val, alternative="greater")   # H1: removed > valid
            rows.append({"Entity": entity, "Metric": col,
                         "median_removed": round(rem.median(), 2),
                         "median_valid":   round(val.median(), 2),
                         "MW_U": int(u), "p_removed_gt_valid": p})
    res = pd.DataFrame(rows); display(res)
    res.to_csv(os.path.join(OUTPUT_DIR, "Table_RPIF_CloudCrossCheck.csv"), index=False)

In [ ]:
import os, numpy as np, pandas as pd
from itertools import combinations
GAP_MIN = 40.0
rows = []
df_filtered["Date_only"] = pd.to_datetime(df_filtered["Date"]).dt.date
for (entity, day), g in df_filtered.groupby(["Entity", "Date_only"]):
    imgs = g["Smart_ID"].unique()
    if len(imgs) < 2: continue
    for a, b in combinations(imgs, 2):
        ga, gb = g[g.Smart_ID == a], g[g.Smart_ID == b]
        if abs(ga["True_Solar_Hour"].iloc[0] - gb["True_Solar_Hour"].iloc[0]) * 60 < GAP_MIN:
            continue
        ga, gb = ga.set_index("pixel_id"), gb.set_index("pixel_id")
        common = ga.index.intersection(gb.index)
        if len(common) < 30: continue
        d_raw = (ga.loc[common, "NDVI"] - gb.loc[common, "NDVI"]).values
        keep  = (ga.loc[common, "is_bio_valid"] & gb.loc[common, "is_bio_valid"]).values
        rows.append({"Entity": entity,
                     "RMSE_raw": np.sqrt(np.mean(d_raw**2)),
                     "RMSE_pur": np.sqrt(np.mean(d_raw[keep]**2)) if keep.sum() > 5 else np.nan})
pairs = pd.DataFrame(rows)
agg = (pairs.dropna().groupby("Entity")
       .agg(n_pairs=("RMSE_raw","size"), RMSE_raw=("RMSE_raw","mean"), RMSE_pur=("RMSE_pur","mean")))
agg["RMSE_reduction_%"] = ((agg.RMSE_raw - agg.RMSE_pur) / agg.RMSE_raw * 100).round(1)
display(agg.round(4))
agg.to_csv(os.path.join(OUTPUT_DIR, "Table_RPIF_IntradayRMSE_benefit.csv"))

In [ ]:
import os, numpy as np, pandas as pd
IQR_GRID, SMOOTH_GRID = [0.5, 1.0, 1.5, 2.0], [1, 3, 5]
rows = []
for proj_name, cfg in PROJECTS.items():
    dfp = df_all[df_all["Project"] == proj_name]
    for entity in dfp["Entity"].unique():
        de = dfp[dfp["Entity"] == entity]; sig_tot = de["NDVI"].std()
        for tol in IQR_GRID:
            for sm in SMOOTH_GRID:
                env = build_envelope(de, cfg["years_gold"], iqr_tol=tol, smooth=sm)
                if env is None: continue
                m = de.merge(env[["H_Week","lower_limit"]], on="H_Week", how="left")
                v = m["NDVI"] >= m["lower_limit"]
                rows.append({"Entity": entity, "IQR_tol": tol, "smooth": sm,
                             "Rejection_%": round((~v).mean()*100, 2),
                             "Sigma_red_%": round((sig_tot - m.loc[v,"NDVI"].std())/sig_tot*100, 2)})
sens = pd.DataFrame(rows)
for entity, g in sens.groupby("Entity"):
    print(f"\n{entity} — Rejection_% (filas IQR_tol, col smooth):")
    display(g.pivot(index="IQR_tol", columns="smooth", values="Rejection_%"))
sens.to_csv(os.path.join(OUTPUT_DIR, "Table_RPIF_SensitivityAnalysis.csv"), index=False)

In [ ]:
import os, glob

# Buscar el parquet en el árbol del proyecto
hits = glob.glob(os.path.join(BASE, "00_rpif_output", "Master_RPIF_Filtered.parquet"))
print("Encontrados:")
for h in hits:
    print("  ", h)

In [ ]:
# ============================================================
# VERIFICACIÓN: ¿sobre qué población está calculada la Tabla 4?
# ============================================================
print("=== VERIFICACIÓN TABLA 4 ===\n")
for ent in ['Avocado', 'Corema album', 'Mango']:
    sub_all = df[df['Entity'] == ent]
    sub_fc  = df[(df['Entity'] == ent) & (df['Veg_Fraction'] >= 0.70)]
    
    n_pix_all = sub_all['pixel_id'].nunique()
    n_pix_fc  = sub_fc['pixel_id'].nunique()
    n_obs_all = len(sub_all)
    n_obs_fc  = len(sub_fc)
    n_removed_all = (~sub_all['is_bio_valid']).sum()
    n_removed_fc  = (~sub_fc['is_bio_valid']).sum()
    
    print(f"{ent}:")
    print(f"  TODAS las obs (pre-fc):    {n_obs_all:>10,}  ({n_pix_all} píxeles únicos)")
    print(f"  Obs con fc>=70% (post-fc): {n_obs_fc:>10,}  ({n_pix_fc} píxeles únicos)")
    print(f"  Removed pre-fc:  {n_removed_all:>8,}  ({100*n_removed_all/n_obs_all:.2f}%)")
    print(f"  Removed post-fc: {n_removed_fc:>8,}  ({100*n_removed_fc/n_obs_fc:.2f}%)")
    print()

In [ ]:
# ============================================================
# COMPARACIÓN: valores derivados pre-fc vs post-fc
# Para saber qué cambia en el resto de la Tabla 4
# ============================================================
import numpy as np

for ent in ['Avocado', 'Corema album', 'Mango']:
    print(f"\n{'='*50}")
    print(f"{ent}")
    print(f"{'='*50}")
    
    for label, sub in [
        ('PRE-FC (actual Tabla 4)', df[df['Entity']==ent]),
        ('POST-FC (corregida)', df[(df['Entity']==ent) & (df['Veg_Fraction']>=0.70)])
    ]:
        valid = sub[sub['is_bio_valid']==True]['NDVI']
        removed = sub[sub['is_bio_valid']==False]['NDVI']
        cb_rem = sub[sub['is_bio_valid']==False]['Coastal_Blue'].mean()
        cb_val = sub[sub['is_bio_valid']==True]['Coastal_Blue'].mean()
        sigma_all = sub['NDVI'].std()
        sigma_valid = valid.std()
        sigma_red = 100*(sigma_all-sigma_valid)/sigma_all if sigma_all>0 else np.nan
        
        print(f"\n  {label}:")
        print(f"    Total obs:        {len(sub):>10,}")
        print(f"    Removed:          {len(removed):>10,} ({100*len(removed)/len(sub):.2f}%)")
        print(f"    NDVI valid mean:  {valid.mean():.3f}")
        print(f"    NDVI removed mean:{removed.mean():.3f}")
        print(f"    Sigma reduction:  {sigma_red:.2f}%")
        print(f"    CB removed mean:  {cb_rem:.3f}")
        print(f"    CB valid mean:    {cb_val:.3f}")
        print(f"    CB ratio:         {cb_rem/cb_val:.1f}×")

In [ ]:
# ============================================================
# RECOMPUTE: haze/substrate classification post-fc
# ============================================================
import numpy as np

# Ajusta estos umbrales a los que usaste en tu análisis original
CB_HAZE_THRESHOLD = 0.15      # por encima = bruma
NIR_RED_SOIL_MAX  = 2.0       # por debajo = sustrato (ajusta si necesario)

print("=== CLASIFICACIÓN POST-FC DE OBSERVACIONES ELIMINADAS ===\n")
for ent in ['Avocado', 'Corema album', 'Mango']:
    sub = df[(df['Entity']==ent) & (df['Veg_Fraction']>=0.70)]
    removed = sub[sub['is_bio_valid']==False].copy()
    n = len(removed)
    if n == 0:
        print(f"{ent}: no removed observations post-fc"); continue
    
    haze_mask = removed['Coastal_Blue'] > CB_HAZE_THRESHOLD
    soil_mask = (~haze_mask) & ((removed['NIR']/removed['Red'].replace(0,np.nan)) < NIR_RED_SOIL_MAX)
    
    pct_haze = 100*haze_mask.sum()/n
    pct_soil = 100*soil_mask.sum()/n
    pct_other = 100 - pct_haze - pct_soil
    
    print(f"{ent} (n_removed={n:,}):")
    print(f"  Haze-like:      {pct_haze:.1f}%")
    print(f"  Substrate-like: {pct_soil:.1f}%")
    print(f"  Other:          {pct_other:.1f}%")
    print()

In [ ]:
# ============================================================
# RECOMPUTE: Table 5 sensitivity analysis post-fc
# ============================================================
print("=== TABLA 5 SENSIBILIDAD IQR_tol (POST-FC) ===\n")
print(f"{'IQR_tol':>8} | {'Avocado':>10} | {'Corema':>12} | {'Mango':>8}")
print("-" * 48)

for tol in [0.5, 1.0, 1.5, 2.0]:
    row = []
    for ent in ['Avocado', 'Corema album', 'Mango']:
        sub = df[(df['Entity']==ent) & (df['Veg_Fraction']>=0.70)].copy()
        
        # Recalcular threshold con este tol
        sub['week'] = pd.to_datetime(sub['Date']).dt.to_period('W').dt.start_time
        sub['HW']   = sub['week'].dt.month.map(lambda m: ...) # usa tu función de HW
        
        # Más simple: usar el lower_limit que ya está en el df y ajustar el IQR_tol
        # lower_limit fue calculado con el tol adoptado → necesitas recalcular
        # Alternativamente, comprueba si tienes una función build_envelope en el notebook
        row.append('--')
    print(f"{tol:>8} | {row[0]:>10} | {row[1]:>12} | {row[2]:>8}")

In [ ]:
import pandas as pd
import numpy as np

tol_values = [0.5, 1.0, 1.5, 2.0]
results = {}

for ent in ['Avocado', 'Corema album', 'Mango']:
    results[ent] = {}
    sub = df[(df['Entity']==ent) & (df['Veg_Fraction']>=0.70)].copy()
    
    # Construir envelope post-fc por HW
    envelope = (sub.groupby('H_Week')['NDVI']
                .agg(Q25=lambda x: x.quantile(0.25),
                     IQR=lambda x: x.quantile(0.75) - x.quantile(0.25))
                .reset_index())
    
    for tol in tol_values:
        merged = sub.merge(envelope, on='H_Week', how='left')
        merged['tol_lower'] = merged['Q25'] - tol * merged['IQR']
        removed = (merged['NDVI'] < merged['tol_lower']).sum()
        results[ent][tol] = 100 * removed / len(merged)

print(f"\n{'IQR_tol':>8} | {'Avocado':>10} | {'Corema':>14} | {'Mango':>10}")
print("-" * 52)
for tol in tol_values:
    row = [f"{results[ent][tol]:.2f}%" for ent in ['Avocado','Corema album','Mango']]
    bold = " *" if tol in [1.0, 2.0] else ""
    print(f"{tol:>8} | {row[0]:>10} | {row[1]:>14} | {row[2]:>10}{bold}")

In [ ]:
# ============================================================
# TIME SERIES FIGURE — Raw vs Purified phenological curves
# NDVI and PRI, three covers
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

# --- Cargar el master filtrado (ajusta la ruta) ---
df = pd.read_parquet(os.path.join(BASE, "00_rpif_output", "Master_RPIF_Filtered.parquet"))

# Verifica que existen estas columnas; ajusta nombres si difieren:
#   'Entity', 'Date', 'NDVI', 'PRI_proxy', 'bio_valid' (o flag de RPIF)
# 'bio_valid' = True para observaciones retenidas, False para eliminadas
print(df.columns.tolist())
print(df['Entity'].unique())

entities = ['Corema album', 'Avocado', 'Mango']
indices  = [('NDVI', 'NDVI'), ('PRI_proxy', 'PRI')]

def weekly_median(sub, col):
    """Agrega a mediana semanal y devuelve serie temporal ordenada."""
    s = (sub.dropna(subset=[col])
            .assign(week=lambda d: pd.to_datetime(d['Date']).dt.to_period('W').dt.start_time)
            .groupby('week')[col].median()
            .sort_index())
    return s

def smooth(series, window=11, poly=2):
    """Savitzky-Golay suave; requiere al menos 'window' puntos."""
    if len(series) < window:
        return series
    return pd.Series(savgol_filter(series.values, window, poly), index=series.index)

fig, axes = plt.subplots(len(indices), len(entities), figsize=(16, 8), sharex=True)

for r, (col, label) in enumerate(indices):
    for c, ent in enumerate(entities):
        ax = axes[r, c]
        sub = df[df['Entity'] == ent]

        # RAW: todas las observaciones (válidas + eliminadas)
        raw = weekly_median(sub, col)
        # PURIFIED: solo las retenidas por el RPIF
        pur = weekly_median(sub[sub['is_bio_valid'] == True], col)

        # Nubes de puntos crudos en gris claro
        ax.scatter(raw.index, raw.values, s=6, color='#cccccc',
                   alpha=0.5, label='Raw weekly median', zorder=1)
        # Curva suavizada cruda (línea fina gris)
        ax.plot(raw.index, smooth(raw).values, color='#999999',
                lw=1, ls='--', label='Raw (smoothed)', zorder=2)
        # Curva suavizada purificada (línea gruesa de color)
        col_clean = '#2ca02c' if label == 'NDVI' else '#ff7f0e'
        ax.plot(pur.index, smooth(pur).values, color=col_clean,
                lw=2.2, label='Purified (smoothed)', zorder=3)

        if r == 0:
            ax.set_title(ent, fontsize=12)
        if c == 0:
            ax.set_ylabel(f'{label}')
        ax.grid(alpha=0.3)
        if r == 0 and c == 0:
            ax.legend(fontsize=8, loc='best')

axes[-1, 1].set_xlabel('Date')
plt.suptitle('Phenological time series: raw vs RPIF-purified', fontsize=14, y=1.0)
plt.tight_layout()
plt.savefig('Fig_TimeSeries_RawVsPurified.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved Fig_TimeSeries_RawVsPurified.png")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import savgol_filter

entities = ['Corema album', 'Avocado', 'Mango']
indices  = [('NDVI', 'NDVI'), ('PRI_proxy', 'PRI')]
band_colors = {'NDVI': '#2ca02c', 'PRI': '#e08214'}

def weekly_pct(sub, col):
    g = (sub.dropna(subset=[col])
            .assign(week=lambda d: pd.to_datetime(d['Date']).dt.to_period('W').dt.start_time)
            .groupby('week')[col])
    return pd.DataFrame({'p10': g.quantile(.10), 'p50': g.quantile(.50),
                         'p90': g.quantile(.90)}).sort_index()

def sm(s, window=11, poly=2):
    if len(s) < 5: return s
    w = min(window, len(s));  w -= (w % 2 == 0)
    return pd.Series(savgol_filter(s.values, w, poly), index=s.index) if w >= 5 else s

plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
fig, axes = plt.subplots(2, 3, figsize=(16, 7.5), sharex=True)

for r, (col, label) in enumerate(indices):
    ymins, ymaxs = [], []
    for c, ent in enumerate(entities):
        ax = axes[r, c]
        sub = df[df['Entity'] == ent]
        raw = weekly_pct(sub, col)
        pur = weekly_pct(sub[sub['is_bio_valid']], col)

        rlo, rhi = sm(raw['p10']), sm(raw['p90'])
        plo, phi, pmed = sm(pur['p10']), sm(pur['p90']), sm(pur['p50'])

        # Banda cruda (ancha, gris) — el ruido original
        ax.fill_between(rlo.index, rlo.values, rhi.values, color='#9e9e9e',
                        alpha=.30, lw=0, label='Raw P10–P90', zorder=1)
        # Banda purificada (estrecha, color) — dispersión reducida
        ax.fill_between(plo.index, plo.values, phi.values, color=band_colors[label],
                        alpha=.35, lw=0, label='Purified P10–P90', zorder=2)
        # Mediana purificada
        ax.plot(pmed.index, pmed.values, color=band_colors[label], lw=2.2,
                label='Purified median', zorder=3)

        if r == 0: ax.set_title(ent, fontsize=12, fontweight='bold')
        if c == 0: ax.set_ylabel(label, fontsize=12)
        ax.grid(alpha=.25, lw=.5)
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ymins.append(np.nanmin(rlo.values)); ymaxs.append(np.nanmax(rhi.values))

    lo, hi = min(ymins), max(ymaxs)
    for c in range(3):
        axes[r, c].set_ylim(lo - .02, hi + .02)

axes[0, 0].legend(fontsize=8, loc='upper left', framealpha=.9)
fig.text(0.5, 0.02, 'Date', ha='center', fontsize=12)
fig.suptitle('Phenological dispersion before and after RPIF purification',
             fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.savefig('Fig_TimeSeries_RawVsPurified.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import savgol_filter

cover_colors = {'Corema album': '#1f77b4', 'Avocado': '#2ca02c', 'Mango': '#ff7f0e'}
entities = list(cover_colors.keys())
indices  = [('NDVI', 'NDVI'), ('PRI_proxy', 'PRI')]
SHOW_BANDS = True   # pon False si el PRI sale muy cargado

def weekly_pct(sub, col):
    g = (sub.dropna(subset=[col])
            .assign(week=lambda d: pd.to_datetime(d['Date']).dt.to_period('W').dt.start_time)
            .groupby('week')[col])
    return pd.DataFrame({'p25': g.quantile(.25), 'p50': g.quantile(.50),
                         'p75': g.quantile(.75)}).sort_index()

def sm(s, window=13, poly=2):
    if len(s) < 5: return s
    w = min(window, len(s)); w -= (w % 2 == 0)
    return pd.Series(savgol_filter(s.values, w, poly), index=s.index) if w >= 5 else s

plt.rcParams.update({'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})
fig, (ax_n, ax_p) = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
panel = {'NDVI': ax_n, 'PRI': ax_p}

for col, label in indices:
    ax = panel[label]
    for ent in entities:
        sub = df[(df['Entity'] == ent) & (df['is_bio_valid'])]
        pct = weekly_pct(sub, col)
        med, lo, hi = sm(pct['p50']), sm(pct['p25']), sm(pct['p75'])
        if SHOW_BANDS:
            ax.fill_between(lo.index, lo.values, hi.values,
                            color=cover_colors[ent], alpha=.12, lw=0, zorder=1)
        ax.plot(med.index, med.values, color=cover_colors[ent],
                lw=2.4, label=ent, zorder=3)
    ax.set_ylabel(label, fontsize=13, fontweight='bold')
    ax.grid(alpha=.25, lw=.5)
    ax.margins(x=0.01)

ax_n.legend(loc='upper left', framealpha=.92, ncol=3, fontsize=11)
ax_p.xaxis.set_major_locator(mdates.YearLocator())
ax_p.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax_p.set_xlabel('Date', fontsize=13)
fig.suptitle('Purified phenological time series by vegetation cover',
             fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('Fig_TimeSeries_Purified_byCover.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from scipy.signal import savgol_filter

cover_colors = {'Corema album': '#1f77b4', 'Avocado': '#2ca02c', 'Mango': '#ff7f0e'}
entities = list(cover_colors.keys())
indices  = [('NDVI', 'NDVI'), ('PRI_proxy', 'PRI')]
STAT = 'median'   # 'mean' hace visible la contaminación; 'median' = robusto (raw≈purified)

def weekly_stat(sub, col, stat):
    g = (sub.dropna(subset=[col])
            .assign(week=lambda d: pd.to_datetime(d['Date']).dt.to_period('W').dt.start_time)
            .groupby('week')[col])
    return (g.mean() if stat == 'mean' else g.median()).sort_index()

def sm(s, window=13, poly=2):
    if len(s) < 5: return s
    w = min(window, len(s)); w -= (w % 2 == 0)
    return pd.Series(savgol_filter(s.values, w, poly), index=s.index) if w >= 5 else s

plt.rcParams.update({'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})
fig, (ax_n, ax_p) = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
panel = {'NDVI': ax_n, 'PRI': ax_p}

for col, label in indices:
    ax = panel[label]
    for ent in entities:
        sub = df[df['Entity'] == ent]
        raw = sm(weekly_stat(sub, col, STAT))
        pur = sm(weekly_stat(sub[sub['is_bio_valid']], col, STAT))
        # Cruda (sin filtrar): fina, discontinua, tenue
        ax.plot(raw.index, raw.values, color=cover_colors[ent], lw=1.0,
                ls='--', alpha=.55, zorder=2)
        # Purificada: gruesa, sólida
        ax.plot(pur.index, pur.values, color=cover_colors[ent], lw=2.4,
                solid_capstyle='round', zorder=3, label=ent)
    ax.set_ylabel(label, fontsize=13, fontweight='bold')
    ax.grid(alpha=.25, lw=.5); ax.margins(x=0.01)

leg1 = ax_n.legend(loc='upper left', framealpha=.92, ncol=3, fontsize=11)
ax_n.add_artist(leg1)
style_handles = [Line2D([0],[0], color='grey', lw=1.0, ls='--', alpha=.7, label='Raw (unfiltered)'),
                 Line2D([0],[0], color='grey', lw=2.4, label='Purified')]
ax_p.legend(handles=style_handles, loc='lower left', framealpha=.92, fontsize=10)

ax_p.xaxis.set_major_locator(mdates.YearLocator())
ax_p.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax_p.set_xlabel('Date', fontsize=13)
fig.suptitle(f'Phenological time series by cover: raw vs purified ({STAT})',
             fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('Fig_TimeSeries_RawVsPurified_byCover.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from scipy.signal import savgol_filter

cover_colors = {'Corema album': '#1f77b4', 'Avocado': '#2ca02c', 'Mango': '#ff7f0e'}
entities = list(cover_colors.keys())

def weekly_median(sub, col):
    return (sub.dropna(subset=[col])
               .assign(week=lambda d: pd.to_datetime(d['Date'])
                                        .dt.to_period('W').dt.start_time)
               .groupby('week')[col].median()
               .sort_index())

def sm(s, window=13, poly=2):
    if len(s) < 5: return s
    w = min(window, len(s)); w -= (w % 2 == 0)
    return pd.Series(savgol_filter(s.values, w, poly), index=s.index) if w >= 5 else s

# --- Determinar la fecha de inicio común (primera semana en que los 3 tienen datos) ---
start_dates = []
for ent in entities:
    sub = df[(df['Entity'] == ent) & (df['is_bio_valid'])]
    s = weekly_median(sub, 'NDVI')
    start_dates.append(s.index.min())
common_start = max(start_dates)   # primera semana donde TODOS tienen datos
print(f"Inicio común: {common_start.date()}")

plt.rcParams.update({'font.size': 12,
                     'axes.spines.top': False,
                     'axes.spines.right': False})
fig, ax = plt.subplots(figsize=(15, 5.5))

for ent in entities:
    sub = df[df['Entity'] == ent]

    # Series crudas y purificadas, recortadas al inicio común
    raw = sm(weekly_median(sub, 'NDVI'))
    pur = sm(weekly_median(sub[sub['is_bio_valid']], 'NDVI'))
    raw = raw[raw.index >= common_start]
    pur = pur[pur.index >= common_start]

    # Cruda: fina, discontinua, tenue
    ax.plot(raw.index, raw.values, color=cover_colors[ent],
            lw=1.0, ls='--', alpha=.50, zorder=2)
    # Purificada: gruesa, sólida
    ax.plot(pur.index, pur.values, color=cover_colors[ent],
            lw=2.4, solid_capstyle='round', zorder=3, label=ent)

ax.set_ylabel('NDVI', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontsize=13)
ax.grid(alpha=.25, lw=.5)
ax.margins(x=0.01)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Leyenda coberturas + estilos
leg1 = ax.legend(loc='upper left', framealpha=.92, ncol=3, fontsize=11)
ax.add_artist(leg1)
style_handles = [
    Line2D([0],[0], color='grey', lw=1.0, ls='--', alpha=.6, label='Raw (unfiltered)'),
    Line2D([0],[0], color='grey', lw=2.4,             label='Purified')
]
ax.legend(handles=style_handles, loc='lower right', framealpha=.92, fontsize=10)

fig.suptitle('Purified vs raw NDVI phenological time series by vegetation cover',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Fig_TimeSeries_NDVI_Final.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D

cover_colors = {'Corema album': '#1f77b4', 'Avocado': '#2ca02c', 'Mango': '#ff7f0e'}
entities = list(cover_colors.keys())

def weekly_median(sub, col):
    return (sub.dropna(subset=[col])
               .assign(week=lambda d: pd.to_datetime(d['Date'])
                                        .dt.to_period('W').dt.start_time)
               .groupby('week')[col].median()
               .sort_index())

# --- Inicio común ---
start_dates = []
for ent in entities:
    s = weekly_median(df[(df['Entity']==ent) & (df['is_bio_valid'])], 'NDVI')
    start_dates.append(s.index.min())
common_start = max(start_dates)
print(f"Inicio común: {common_start.date()}")

plt.rcParams.update({'font.size': 12,
                     'axes.spines.top': False,
                     'axes.spines.right': False})
fig, ax = plt.subplots(figsize=(15, 5.5))

for ent in entities:
    sub = df[df['Entity'] == ent]
    raw = weekly_median(sub, 'NDVI')
    pur = weekly_median(sub[sub['is_bio_valid']], 'NDVI')
    raw = raw[raw.index >= common_start]
    pur = pur[pur.index >= common_start]

    # Sin suavizado — serie semanal real
    ax.plot(raw.index, raw.values, color=cover_colors[ent],
            lw=0.8, ls='--', alpha=.45, zorder=2)
    ax.plot(pur.index, pur.values, color=cover_colors[ent],
            lw=1.6, solid_capstyle='round', zorder=3, label=ent)

ax.set_ylabel('NDVI', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontsize=13)
ax.grid(alpha=.25, lw=.5)
ax.margins(x=0.01)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

leg1 = ax.legend(loc='upper left', framealpha=.92, ncol=3, fontsize=11)
ax.add_artist(leg1)
style_handles = [
    Line2D([0],[0], color='grey', lw=0.8, ls='--', alpha=.55, label='Raw (unfiltered)'),
    Line2D([0],[0], color='grey', lw=1.6, label='Purified')
]
ax.legend(handles=style_handles, loc='lower right', framealpha=.92, fontsize=10)

fig.suptitle('Purified vs raw NDVI — unsmoothed weekly medians',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Fig_TimeSeries_NDVI_Unsmoothed.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from scipy.signal import savgol_filter

cover_colors = {'Corema album': '#1f77b4', 'Avocado': '#2ca02c', 'Mango': '#ff7f0e'}
entities = list(cover_colors.keys())

def weekly_median(sub, col):
    return (sub.dropna(subset=[col])
               .assign(week=lambda d: pd.to_datetime(d['Date'])
                                        .dt.to_period('W').dt.start_time)
               .groupby('week')[col].median()
               .sort_index())

def sm(s, window=13, poly=2):
    if len(s) < 5: return s
    w = min(window, len(s)); w -= (w % 2 == 0)
    return pd.Series(savgol_filter(s.values, w, poly), index=s.index) if w >= 5 else s

# Inicio común
start_dates = []
for ent in entities:
    s = weekly_median(df[(df['Entity']==ent) & (df['is_bio_valid'])], 'NDVI')
    start_dates.append(s.index.min())
common_start = max(start_dates)

plt.rcParams.update({'font.size': 12,
                     'axes.spines.top': False,
                     'axes.spines.right': False})
fig, ax = plt.subplots(figsize=(15, 5.5))

all_values = []
for ent in entities:
    sub = df[df['Entity'] == ent]
    raw = sm(weekly_median(sub, 'NDVI'))
    pur = sm(weekly_median(sub[sub['is_bio_valid']], 'NDVI'))
    raw = raw[raw.index >= common_start]
    pur = pur[pur.index >= common_start]

    ax.plot(raw.index, raw.values, color=cover_colors[ent],
            lw=1.0, ls='--', alpha=.50, zorder=2)
    ax.plot(pur.index, pur.values, color=cover_colors[ent],
            lw=2.4, solid_capstyle='round', zorder=3, label=ent)

    all_values.extend(raw.values.tolist())
    all_values.extend(pur.values.tolist())

# Zoom al rango real de los datos + margen pequeño
data_min = np.nanmin(all_values)
data_max = np.nanmax(all_values)
margin = (data_max - data_min) * 0.05   # 5% de margen arriba y abajo
ax.set_ylim(data_min - margin, data_max + margin)

ax.set_ylabel('NDVI', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontsize=13)
ax.grid(alpha=.25, lw=.5)
ax.margins(x=0.01)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

leg1 = ax.legend(loc='upper left', framealpha=.92, ncol=3, fontsize=11)
ax.add_artist(leg1)
style_handles = [
    Line2D([0],[0], color='grey', lw=1.0, ls='--', alpha=.6, label='Raw (unfiltered)'),
    Line2D([0],[0], color='grey', lw=2.4, label='Purified')
]
ax.legend(handles=style_handles, loc='lower right', framealpha=.92, fontsize=10)

fig.suptitle('Purified vs raw NDVI phenological time series by vegetation cover',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Fig_TimeSeries_NDVI_Final.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import savgol_filter

bands = ['Green_I', 'Green', 'Yellow']
band_colors = {'Green_I': '#2ca02c', 'Green': '#17becf', 'Yellow': '#e8a020'}
entities_ax = {'Avocado': 0, 'Mango': 1}

def weekly_median(sub, col):
    return (sub.dropna(subset=[col])
               .assign(week=lambda d: pd.to_datetime(d['Date'])
                                        .dt.to_period('W').dt.start_time)
               .groupby('week')[col].median()
               .sort_index())

def sm(s, window=13, poly=2):
    if len(s) < 5: return s
    w = min(window, len(s)); w -= (w % 2 == 0)
    return pd.Series(savgol_filter(s.values, w, poly), index=s.index) if w >= 5 else s

plt.rcParams.update({'font.size': 12,
                     'axes.spines.top': False,
                     'axes.spines.right': False})
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

for ent, row in entities_ax.items():
    ax = axes[row]
    sub = df[(df['Entity'] == ent) & (df['is_bio_valid'])]
    all_vals = []

    for band in bands:
        s = sm(weekly_median(sub, band))
        ax.plot(s.index, s.values, color=band_colors[band],
                lw=2.2, solid_capstyle='round', label=f'{band}')
        all_vals.extend(s.dropna().values.tolist())

    # Zoom al rango real
    mn, mx = np.nanmin(all_vals), np.nanmax(all_vals)
    margin = (mx - mn) * 0.05
    ax.set_ylim(mn - margin, mx + margin)

    ax.set_title(ent, fontsize=13, fontweight='bold')
    ax.set_ylabel('Surface reflectance', fontsize=11)
    ax.grid(alpha=.25, lw=.5)
    ax.margins(x=0.01)
    ax.legend(loc='upper left', framealpha=.92, ncol=3, fontsize=11)

axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[1].set_xlabel('Date', fontsize=12)

fig.suptitle('Green I, Green and Yellow band reflectance — purified pixels',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Fig_TimeSeries_GreenBands_AvoMango.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved.")

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import savgol_filter

BAND_COLS = [
    ("Coastal_Blue", "Coastal"),
    ("Blue",         "Blue"),
    ("Green_I",      "Green I"),
    ("Green",        "Green"),
    ("Yellow",       "Yellow"),
    ("Red",          "Red"),
    ("RedEdge",      "RedEdge"),
    ("NIR",          "NIR"),
]

# Paleta espectral ordenada por longitud de onda
band_colors = {
    "Coastal":  "#7b2d8b",
    "Blue":     "#1f77b4",
    "Green I":  "#2ca02c",
    "Green":    "#17becf",
    "Yellow":   "#e8a020",
    "Red":      "#d62728",
    "RedEdge":  "#8c564b",
    "NIR":      "#aec7e8",
}

entities_ax = {'Avocado': 0, 'Mango': 1}

def weekly_median(sub, col):
    return (sub.dropna(subset=[col])
               .assign(week=lambda d: pd.to_datetime(d['Date'])
                                        .dt.to_period('W').dt.start_time)
               .groupby('week')[col].median()
               .sort_index())

def sm(s, window=13, poly=2):
    if len(s) < 5: return s
    w = min(window, len(s)); w -= (w % 2 == 0)
    return pd.Series(savgol_filter(s.values, w, poly), index=s.index) if w >= 5 else s

plt.rcParams.update({'font.size': 12,
                     'axes.spines.top': False,
                     'axes.spines.right': False})
fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)

for ent, row in entities_ax.items():
    ax = axes[row]
    sub = df[(df['Entity'] == ent) & (df['is_bio_valid'])]
    all_vals = []

    for col, label in BAND_COLS:
        if col not in df.columns:
            print(f"  Columna no encontrada: {col}")
            continue
        s = sm(weekly_median(sub, col))
        ax.plot(s.index, s.values, color=band_colors[label],
                lw=2.0, solid_capstyle='round', label=label)
        all_vals.extend(s.dropna().values.tolist())

    mn, mx = np.nanmin(all_vals), np.nanmax(all_vals)
    margin = (mx - mn) * 0.05
    ax.set_ylim(mn - margin, mx + margin)

    ax.set_title(ent, fontsize=13, fontweight='bold')
    ax.set_ylabel('Surface reflectance', fontsize=11)
    ax.grid(alpha=.25, lw=.5)
    ax.margins(x=0.01)
    ax.legend(loc='upper left', framealpha=.92, ncol=4, fontsize=10)

axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[1].set_xlabel('Date', fontsize=12)

fig.suptitle('All-band surface reflectance time series — purified pixels (Avocado & Mango)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Fig_TimeSeries_AllBands_AvoMango.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved.")

Las bandas de Planet Scope:
Band 3 = Green I	
n/a

Green I: 513 - 549 nm
Band 4 = Green	
n/a

Green: 547 - 583 nm
Band 5 = Yellow	
n/a

Yellow: 600 - 620 nm
Band 6 = Red	
n/a

Red: 650 - 680 nm

In [ ]:
# ============================================================
# BLOQUE 1 — Cálculo de métricas de forma espectral
# ============================================================
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import savgol_filter, find_peaks

WL = {'Coastal_Blue':443, 'Blue':490, 'Green_I':531, 'Green':565,
      'Yellow':610, 'Red':665, 'RedEdge':705, 'NIR':865}
ALLB = list(WL.keys())

# Ventanas de floración aproximadas en Axarquía — AJUSTA a tu conocimiento agronómico
FLOWER = {'Avocado': (3, 5), 'Mango': (2, 4)}   # (mes_inicio, mes_fin)
flower_colors = {'Avocado': '#2ca02c', 'Mango': '#ff7f0e'}

def weekly_median_bands(sub):
    return (sub.assign(week=lambda d: pd.to_datetime(d['Date']).dt.to_period('W').dt.start_time)
               .groupby('week')[ALLB].median().sort_index())

def continuum_depth(row, b_lo='Coastal_Blue', b_hi='RedEdge'):
    lo, hi = WL[b_lo], WL[b_hi]
    r_lo, r_hi = row[b_lo], row[b_hi]
    out = {}
    for b in ['Blue','Green_I','Green','Yellow','Red']:
        cont = r_lo + (r_hi - r_lo) * (WL[b]-lo)/(hi-lo)
        out[f'depth_{b}'] = row[b] - cont
    return pd.Series(out)

def sm(s, w=9, p=2):
    s = s.dropna()
    if len(s) < 5: return s
    w = min(w, len(s)); w -= (w % 2 == 0)
    return pd.Series(savgol_filter(s.values, w, p), index=s.index) if w >= 5 else s

results = {}
for ent in ['Avocado','Mango']:
    sub = df[(df['Entity']==ent) & (df['is_bio_valid'])].copy()
    wk = weekly_median_bands(sub)

    # 1) Normalización por brillo (quita el modo común)
    rel = wk[ALLB].div(wk[ALLB].sum(axis=1), axis=0)

    # 2) Continuum removal sobre el espectro relativo
    depths = rel.apply(continuum_depth, axis=1)

    # 3) Índice de floración: enriquecimiento amarillo/rojo + supresión del pico verde
    depths['FloweringIdx'] = depths['depth_Yellow'] + depths['depth_Red'] - depths['depth_Green']
    results[ent] = depths

print("Métricas:", [c for c in results['Avocado'].columns])

In [ ]:
# ============================================================
# BLOQUE 2 — Climatología fenológica (superposición de años)
# ¿Hay un pico recurrente que coincida con la floración?
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for row, ent in enumerate(['Avocado','Mango']):
    ax = axes[row]
    idx = sm(results[ent]['FloweringIdx'])
    doy = pd.Series([d.dayofyear for d in idx.index], index=idx.index)
    yr  = pd.Series([d.year for d in idx.index], index=idx.index)

    for y in sorted(yr.unique()):
        m = yr == y
        ax.plot(doy[m], idx[m], lw=1.6, alpha=.85, label=str(y))

    m0, m1 = FLOWER[ent]
    d0, d1 = pd.Timestamp(2001,m0,1).dayofyear, pd.Timestamp(2001,m1,28).dayofyear
    ax.axvspan(d0, d1, color=flower_colors[ent], alpha=.15,
               label=f'Flowering ({m0:02d}–{m1:02d})')

    ax.set_title(f'{ent} — visible-shape anomaly by day of year', fontweight='bold')
    ax.set_ylabel('Flowering index'); ax.grid(alpha=.25, lw=.5)
    ax.legend(ncol=4, fontsize=8, loc='upper right')

axes[1].set_xlabel('Day of year'); axes[1].set_xlim(1, 366)
fig.suptitle('Phenological climatology: recurring spectral anomaly vs flowering window',
             fontsize=14, fontweight='bold', y=1.0)
plt.tight_layout()
plt.savefig('Fig_Flowering_Climatology.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
# BLOQUE 3 — Serie completa + detección de picos + test estadístico
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
peak_rows = []
for row, ent in enumerate(['Avocado','Mango']):
    ax = axes[row]
    idx = sm(results[ent]['FloweringIdx'])
    sd  = np.nanstd(idx.values)
    peaks, _ = find_peaks(idx.values, prominence=0.5*sd, distance=20)

    ax.plot(idx.index, idx.values, color=flower_colors[ent], lw=1.8)
    ax.axhline(np.nanmedian(idx.values), color='grey', ls=':', lw=1)

    for pk in peaks:
        d = idx.index[pk]
        in_win = FLOWER[ent][0] <= d.month <= FLOWER[ent][1]
        ax.plot(d, idx.values[pk], 'v', color='k' if in_win else 'grey', ms=7)
        peak_rows.append({'Entity':ent, 'Date':d.date(),
                          'Idx':round(idx.values[pk],4), 'In_window':in_win})

    for y in range(idx.index.year.min(), idx.index.year.max()+1):
        m0, m1 = FLOWER[ent]
        ax.axvspan(pd.Timestamp(y,m0,1), pd.Timestamp(y,m1,28),
                   color=flower_colors[ent], alpha=.10)

    ax.set_title(f'{ent} — flowering index with detected peaks (▼ black = within window)',
                 fontweight='bold')
    ax.set_ylabel('Flowering index'); ax.grid(alpha=.25, lw=.5)

axes[1].xaxis.set_major_locator(mdates.YearLocator())
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('Fig_Flowering_TimeSeries.png', dpi=300, bbox_inches='tight'); plt.show()

peak_df = pd.DataFrame(peak_rows)
print(peak_df.to_string(index=False))
for ent in ['Avocado','Mango']:
    s = peak_df[peak_df['Entity']==ent]
    if len(s):
        print(f"{ent}: {s['In_window'].sum()}/{len(s)} picos dentro de la ventana de floración "
              f"({100*s['In_window'].mean():.0f}%)")

In [ ]:
# ============================================================
# CELDA A — Índices de floración por diferencia normalizada (YFI)
# ============================================================
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.signal import savgol_filter, find_peaks
from scipy.stats import median_abs_deviation

EPS = 1e-6
ALLB = ['Coastal_Blue','Blue','Green_I','Green','Yellow','Red','RedEdge','NIR']

def weekly_median_bands(sub):
    return (sub.assign(week=lambda d: pd.to_datetime(d['Date']).dt.to_period('W').dt.start_time)
               .groupby('week')[ALLB].median().sort_index())

def sm(s, w=9, p=2):
    s = s.dropna()
    if len(s) < 5: return s
    w = min(w, len(s)); w -= (w % 2 == 0)
    return pd.Series(savgol_filter(s.values, w, p), index=s.index) if w >= 5 else s

yfi = {}
for ent in ['Avocado','Mango']:
    wk = weekly_median_bands(df[(df['Entity']==ent) & (df['is_bio_valid'])])
    # Aguacate: flores amarillo-verdosas -> diferencia Green / Green I
    wk['YFI_GG'] = (wk['Green']  - wk['Green_I']) / (wk['Green']  + wk['Green_I'] + EPS)
    # Mango: panículas más anaranjadas -> diferencia Yellow / Green I
    wk['YFI_YG'] = (wk['Yellow'] - wk['Green_I']) / (wk['Yellow'] + wk['Green_I'] + EPS)
    yfi[ent] = wk

print("YFI calculado para Avocado y Mango (YFI_GG y YFI_YG).")

In [ ]:
# ============================================================
# CELDA B — Detección CIEGA de eventos + cronología emergente
# Sin ventana impuesta: los picos y su agrupación por día del año
# revelan cuándo, a priori, hay una señal tipo floración.
# ============================================================
PRIMARY  = {'Avocado': 'YFI_GG', 'Mango': 'YFI_YG'}  # índice principal por cultivo
crop_col = {'Avocado': '#2ca02c', 'Mango': '#ff7f0e'}
K_PROM   = 2.0    # prominencia mínima en múltiplos de MAD (robusto). Sube para ser más estricto
MIN_DIST = 18     # separación mínima entre picos (semanas) ~ evita doble conteo anual

detections = []
clim_peak = {}

fig, axes = plt.subplots(2, 2, figsize=(16, 9),
                         gridspec_kw={'width_ratios':[3, 1]})
for row, ent in enumerate(['Avocado','Mango']):
    s = sm(yfi[ent][PRIMARY[ent]])
    mad = median_abs_deviation(s.values, nan_policy='omit')
    peaks, _ = find_peaks(s.values, prominence=K_PROM*mad, distance=MIN_DIST)

    # ---- Panel izquierdo: serie temporal con picos detectados ----
    axL = axes[row, 0]
    axL.plot(s.index, s.values, color=crop_col[ent], lw=1.8)
    axL.axhline(np.nanmedian(s.values), color='grey', ls=':', lw=1)
    for pk in peaks:
        d = s.index[pk]
        axL.plot(d, s.values[pk], 'v', color='k', ms=8, zorder=5)
        detections.append({'Entity':ent, 'Index':PRIMARY[ent], 'Date':d.date(),
                           'DOY':d.dayofyear, 'YFI':round(s.values[pk],4)})
    axL.set_title(f'{ent} — {PRIMARY[ent]} with blind-detected peaks', fontweight='bold')
    axL.set_ylabel(PRIMARY[ent]); axL.grid(alpha=.25, lw=.5)
    axL.xaxis.set_major_locator(mdates.YearLocator())
    axL.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    # ---- Panel derecho: agrupación por día del año (recurrencia) ----
    axR = axes[row, 1]
    det_ent = [x for x in detections if x['Entity']==ent]
    doys = [x['DOY'] for x in det_ent]
    axR.scatter(doys, [x['YFI'] for x in det_ent], color=crop_col[ent], s=60, zorder=5)
    if doys:
        mu, sigma = np.mean(doys), np.std(doys)
        axR.axvspan(mu-sigma, mu+sigma, color=crop_col[ent], alpha=.15)
        axR.axvline(mu, color='k', ls='--', lw=1)
        # Climatología: pico de la curva mediana por semana-del-año
        woy = pd.Series([d.dayofyear for d in s.index], index=s.index)
        clim = pd.DataFrame({'doy':woy.values, 'val':s.values}).groupby(
                pd.cut(woy.values, bins=np.arange(1,373,7)))['val'].median()
        clim_peak[ent] = (mu, sigma)
        approx = (pd.Timestamp('2001-01-01') + pd.Timedelta(days=int(mu)-1)).strftime('%d %b')
        axR.set_title(f'DOY clustering\n≈ {approx} (±{sigma:.0f} d)', fontsize=10)
    axR.set_xlim(1, 366); axR.set_xlabel('Day of year')
    axR.grid(alpha=.25, lw=.5)

fig.suptitle('Blind spectral detection of candidate flowering events',
             fontsize=14, fontweight='bold', y=1.0)
plt.tight_layout()
plt.savefig('Fig_Flowering_BlindDetection.png', dpi=300, bbox_inches='tight'); plt.show()

# Resumen de la cronología emergente
for ent in ['Avocado','Mango']:
    mu, sigma = clim_peak.get(ent, (np.nan, np.nan))
    approx = (pd.Timestamp('2001-01-01') + pd.Timedelta(days=int(mu)-1)).strftime('%d %B') if mu==mu else '—'
    print(f"{ent}: pico espectral recurrente ≈ DOY {mu:.0f} ({approx}), dispersión ±{sigma:.0f} días")

In [ ]:
# ============================================================
# CELDA C — Tabla para el investigador agrónomo (verdad de campo)
# ============================================================
det_df = pd.DataFrame(detections).sort_values(['Entity','Date'])
det_df['Year'] = pd.to_datetime(det_df['Date']).dt.year
det_df = det_df[['Entity','Year','Date','DOY','Index','YFI']]

det_df.to_csv('candidate_flowering_events.csv', index=False)
print(det_df.to_string(index=False))
print("\nGuardado: candidate_flowering_events.csv")
print("\n-> Pásale esta tabla al investigador. La columna 'Date' es la fecha")
print("   donde el espectro sugiere, a ciegas, un evento tipo floración.")
print("   Él puede confirmar/desmentir contra las fechas reales de su finca.")